<a href="https://colab.research.google.com/github/rt8294820-ux/Bus-Trip-Delay-Analysis/blob/main/notebooks/capstone_project_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files

In [ ]:
uploaded = files.upload()

df = pd.read_csv("bus_trip_delays_dataset.csv")

Saving bus_trip_delays_dataset.csv to bus_trip_delays_dataset.csv


In [ ]:
df.head()

,trip_id,date,day_of_week,route,bus_id,origin,destination,scheduled_departure,actual_departure,scheduled_arrival,actual_arrival,weather,traffic_level,passenger_count
0,T0001,2026-01-01,Thursday,R6,BUS035,University,Shopping Mall,07:00,07:01,07:38,07:39,Cloudy,Low,13
1,T0002,2026-01-01,Thursday,R5,BUS002,Airport Road,City Centre,13:00,12:58,13:55,14:07,Clear,Low,43
2,T0003,2026-01-01,Thursday,R2,BUS001,Railway Station,Tech Park,19:50,19:48,20:40,20:55,Cloudy,Medium,18
3,T0004,2026-01-01,Thursday,R6,BUS007,University,Shopping Mall,13:20,13:20,13:58,14:04,Clear,Low,13
4,T0005,2026-01-01,Thursday,R4,BUS025,Residential Area,Hospital,07:20,07:21,07:50,07:52,Cloudy,Medium,13


In [ ]:
print("Dataset Shape:")
print(df.shape)

Dataset Shape:
(600, 14)


In [ ]:
print("Dataset Information:")
df.info()

Dataset Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 600 entries, 0 to 599
Data columns (total 14 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   trip_id              600 non-null    object
 1   date                 600 non-null    object
 2   day_of_week          600 non-null    object
 3   route                600 non-null    object
 4   bus_id               600 non-null    object
 5   origin               600 non-null    object
 6   destination          600 non-null    object
 7   scheduled_departure  600 non-null    object
 8   actual_departure     600 non-null    object
 9   scheduled_arrival    600 non-null    object
 10  actual_arrival       600 non-null    object
 11  weather              600 non-null    object
 12  traffic_level        600 non-null    object
 13  passenger_count      600 non-null    int64 
dtypes: int64(1), object(13)
memory usage: 65.8+ KB


In [ ]:
print("Column Names:")
print(df.columns)

Column Names:
Index(['trip_id', 'date', 'day_of_week', 'route', 'bus_id', 'origin',
       'destination', 'scheduled_departure', 'actual_departure',
       'scheduled_arrival', 'actual_arrival', 'weather', 'traffic_level',
       'passenger_count'],
      dtype='object')


In [ ]:
print("Statistical Summary:")
df.describe()

Statistical Summary:


,passenger_count
count,600.000000
mean,31.731667
std,13.829984
min,8.000000
25%,19.000000
50%,32.000000
75%,43.000000
max,55.000000


In [ ]:
print("Missing Values:")
print(df.isnull().sum())

Missing Values:
trip_id                0
date                   0
day_of_week            0
route                  0
bus_id                 0
origin                 0
destination            0
scheduled_departure    0
actual_departure       0
scheduled_arrival      0
actual_arrival         0
weather                0
traffic_level          0
passenger_count        0
dtype: int64


In [ ]:
print("Duplicate Records:")
print(df.duplicated().sum())

Duplicate Records:
0


In [ ]:
# Remove duplicate records
df = df.drop_duplicates()

# Remove rows containing missing values
df = df.dropna()

# Convert date column into datetime format
df["date"] = pd.to_datetime(df["date"])

print("Data after cleaning:")
print(df.head())

print("\nDataset Shape:", df.shape)

Data after cleaning:
  trip_id       date day_of_week route  bus_id            origin  \
0   T0001 2026-01-01    Thursday    R6  BUS035        University   
1   T0002 2026-01-01    Thursday    R5  BUS002      Airport Road   
2   T0003 2026-01-01    Thursday    R2  BUS001   Railway Station   
3   T0004 2026-01-01    Thursday    R6  BUS007        University   
4   T0005 2026-01-01    Thursday    R4  BUS025  Residential Area   

     destination scheduled_departure actual_departure scheduled_arrival  \
0  Shopping Mall               07:00            07:01             07:38   
1    City Centre               13:00            12:58             13:55   
2      Tech Park               19:50            19:48             20:40   
3  Shopping Mall               13:20            13:20             13:58   
4       Hospital               07:20            07:21             07:50   

  actual_arrival weather traffic_level  passenger_count  
0          07:39  Cloudy           Low               13  
1  

In [ ]:
# Convert scheduled and actual times into datetime format

df["scheduled_arrival"] = pd.to_datetime(
    df["date"].astype(str) + " " + df["scheduled_arrival"]
)

df["actual_arrival"] = pd.to_datetime(
    df["date"].astype(str) + " " + df["actual_arrival"]
)

df["scheduled_departure"] = pd.to_datetime(
    df["date"].astype(str) + " " + df["scheduled_departure"]
)

df["actual_departure"] = pd.to_datetime(
    df["date"].astype(str) + " " + df["actual_departure"]
)

print("Time columns converted successfully.")

Time columns converted successfully.


In [ ]:
# Calculate arrival delay in minutes
df["arrival_delay_minutes"] = (
    df["actual_arrival"] - df["scheduled_arrival"]
).dt.total_seconds() / 60

# Calculate departure delay in minutes
df["departure_delay_minutes"] = (
    df["actual_departure"] - df["scheduled_departure"]
).dt.total_seconds() / 60

# Count only positive delay minutes
df["delay_minutes"] = df["arrival_delay_minutes"].clip(lower=0)

df[[
    "route",
    "arrival_delay_minutes",
    "departure_delay_minutes",
    "delay_minutes"
]].head()

,route,arrival_delay_minutes,departure_delay_minutes,delay_minutes
0,R6,1.0,1.0,1.0
1,R5,12.0,-2.0,12.0
2,R2,15.0,-2.0,15.0
3,R6,6.0,0.0,6.0
4,R4,2.0,1.0,2.0


In [ ]:
df["delay_status"] = np.where(
    df["arrival_delay_minutes"] > 0,
    "Delayed",
    "On Time or Early"
)

print(df["delay_status"].value_counts())

delay_status
Delayed             548
On Time or Early     52
Name: count, dtype: int64


In [ ]:
total_trips = len(df)

delayed_trips = (df["delay_minutes"] > 0).sum()

average_delay = df["delay_minutes"].mean()

maximum_delay = df["delay_minutes"].max()

print("Total Trips:", total_trips)
print("Delayed Trips:", delayed_trips)
print("Average Delay:", round(average_delay, 2), "minutes")
print("Maximum Delay:", maximum_delay, "minutes")

delay_percentage = (delayed_trips / total_trips) * 100

print("Delay Percentage:", round(delay_percentage, 2), "%")

Total Trips: 600
Delayed Trips: 548
Average Delay: 9.68 minutes
Maximum Delay: 28.0 minutes
Delay Percentage: 91.33 %


In [ ]:
route_analysis = df.groupby("route")["delay_minutes"].mean()

print("Average Delay by Route:")
print(route_analysis.sort_values(ascending=False))

Average Delay by Route:
route
R2    10.596154
R1    10.024691
R3     9.604396
R6     9.477064
R5     9.288462
R4     9.207207
Name: delay_minutes, dtype: float64


In [ ]:
hour = df["scheduled_departure"].dt.hour

df["time_of_day"] = pd.cut(
    hour,
    bins=[0, 12, 17, 24],
    labels=["Morning", "Afternoon", "Evening"],
    right=False
)

time_analysis = df.groupby(
    "time_of_day",
    observed=False
)["delay_minutes"].mean()

print("Average Delay by Time of Day:")
print(time_analysis)

Average Delay by Time of Day:
time_of_day
Morning       9.629108
Afternoon    10.372222
Evening       9.135266
Name: delay_minutes, dtype: float64


In [ ]:
day_analysis = df.groupby(
    "day_of_week"
)["delay_minutes"].mean()

day_order = [
    "Monday", "Tuesday", "Wednesday",
    "Thursday", "Friday", "Saturday", "Sunday"
]

day_analysis = day_analysis.reindex(day_order)

print("Average Delay by Day:")
print(day_analysis)

Average Delay by Day:
day_of_week
Monday       10.600000
Tuesday      11.137500
Wednesday     9.937500
Thursday      8.955556
Friday        9.600000
Saturday      8.955556
Sunday        8.877778
Name: delay_minutes, dtype: float64


In [ ]:
traffic_analysis = df.groupby(
    "traffic_level"
)["delay_minutes"].mean()

print("Average Delay by Traffic Level:")
print(traffic_analysis)

Average Delay by Traffic Level:
traffic_level
High      15.348101
Low        4.851064
Medium     8.970100
Name: delay_minutes, dtype: float64


In [ ]:
weather_analysis = df.groupby(
    "weather"
)["delay_minutes"].mean()

print("Average Delay by Weather:")
print(weather_analysis)

Average Delay by Weather:
weather
Clear      8.595908
Cloudy     9.397163
Rainy     16.514706
Name: delay_minutes, dtype: float64
